In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("PIPELINE MONITORING & DASHBOARDS")
print("=" * 70)


In [0]:

# Cell 1: Create Metrics Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.pipeline_metrics (
    metric_id STRING,
    metric_name STRING,
    metric_value DOUBLE,
    metric_unit STRING,
    metric_category STRING,
    metric_timestamp TIMESTAMP,
    pipeline_run_id STRING,
    status STRING
)
USING DELTA
""")

print("✅ Metrics table created")


In [0]:

# Cell 2: Load Gold Tables
gold_repos = spark.table(f"{catalog}.{schema}.gold_repository_rankings")
gold_contribs = spark.table(f"{catalog}.{schema}.gold_contributor_analysis")
gold_health = spark.table(f"{catalog}.{schema}.gold_ecosystem_health")
gold_languages = spark.table(f"{catalog}.{schema}.gold_language_trends")

print("✅ Loaded gold tables for monitoring")

In [0]:

# Cell 3: Business Metrics
print("\n" + "=" * 70)
print("BUSINESS METRICS")
print("=" * 70 + "\n")

# Metric 1: Total Stars Tracked
total_stars = gold_repos.agg(F.sum("stars")).collect()[0][0]
print(f"✅ Total stars tracked: {total_stars:,}")

# Metric 2: Average Repository Health
avg_health = gold_health.agg(F.avg("health_score")).collect()[0][0]
print(f"✅ Average repository health: {round(avg_health, 2)}")

# Metric 3: Total Contributors
total_contributors = gold_contribs.count()
print(f"✅ Total contributors tracked: {total_contributors}")

# Metric 4: Active Repositories
active_repos = gold_repos.filter(F.col("overall_rank") <= 10).count()
print(f"✅ Top tier repositories: {active_repos}")

# Metric 5: Expert Contributors
expert_contribs = gold_contribs.filter(F.col("expertise_level") == "expert").count()
print(f"✅ Expert contributors: {expert_contribs}")

In [0]:

# Cell 4: Performance Metrics
print("\n" + "=" * 70)
print("PERFORMANCE METRICS")
print("=" * 70 + "\n")

# Metric 1: Average Stars per Repository
avg_stars = gold_repos.agg(F.avg("stars")).collect()[0][0]
print(f"✅ Average stars per repo: {round(avg_stars, 0)}")

# Metric 2: Average Forks
avg_forks = gold_repos.agg(F.avg("forks")).collect()[0][0]
print(f"✅ Average forks per repo: {round(avg_forks, 0)}")

# Metric 3: Contribution Distribution
top_contributor = gold_contribs.orderBy(F.col("total_contributions").desc()).limit(1).collect()[0]
print(f"✅ Top contributor: {top_contributor['contributor_login']} ({int(top_contributor['total_contributions'])} contributions)")

# Metric 4: Health Score Distribution
health_excellent = gold_health.filter(F.col("health_status") == "excellent").count()
health_good = gold_health.filter(F.col("health_status") == "good").count()
health_fair = gold_health.filter(F.col("health_status") == "fair").count()

print(f"\n✅ Health Score Distribution:")
print(f"   Excellent: {health_excellent}")
print(f"   Good: {health_good}")
print(f"   Fair: {health_fair}")

In [0]:

# Cell 5: Trend Analysis
print("\n" + "=" * 70)
print("TREND ANALYSIS")
print("=" * 70 + "\n")

# Most popular language
top_language = gold_languages.orderBy(F.col("total_stars").desc()).limit(1).collect()[0]
print(f"✅ Most popular language: {top_language['language']} ({int(top_language['total_stars'])} stars)")

# Fastest growing language (by momentum)
fastest_growing = gold_languages.orderBy(F.col("momentum").desc()).limit(1).collect()[0]
print(f"✅ Fastest growing language: {fastest_growing['language']} ({round(fastest_growing['momentum'], 2)}% recently updated)")

# Language distribution
print(f"\n✅ Language Distribution:")
display(gold_languages.select("language", "total_repos", "total_stars", "momentum").limit(10))

In [0]:

# Cell 6: Repository Insights
print("\n" + "=" * 70)
print("REPOSITORY INSIGHTS")
print("=" * 70 + "\n")

# Top 10 repositories by stars
print("✅ TOP 10 REPOSITORIES:")
top_10 = gold_repos.orderBy(F.col("overall_rank")).limit(10).select(
    F.col("overall_rank").alias("Rank"),
    F.col("repo_name").alias("Repository"),
    F.col("stars").alias("Stars"),
    F.col("language").alias("Language"),
    F.col("health_score").alias("Health")
)
display(top_10)

In [0]:

# Cell 7: Contributor Insights
print("\n" + "=" * 70)
print("CONTRIBUTOR INSIGHTS")
print("=" * 70 + "\n")

# Top 10 contributors
print("✅ TOP 10 CONTRIBUTORS:")
top_contribs = gold_contribs.orderBy(F.col("total_contributions").desc()).limit(10).select(
    F.col("contributor_login").alias("Contributor"),
    F.col("total_contributions").alias("Total Contributions"),
    F.col("repos_contributed").alias("Repos"),
    F.col("expertise_level").alias("Expertise"),
    F.col("contributor_rank").alias("Rank")
)
display(top_contribs)

In [0]:

# Cell 8: SLA Monitoring
print("\n" + "=" * 70)
print("SLA MONITORING")
print("=" * 70 + "\n")

# SLA 1: Data Freshness (data should be < 24 hours old)
most_recent = spark.table(f"{catalog}.{schema}.bronze_repositories") \
    .select(F.max("ingestion_timestamp")).collect()[0][0]

hours_old = (datetime.now() - most_recent).total_seconds() / 3600 if most_recent else 999

sla_freshness = "PASS" if hours_old < 24 else "FAIL"
print(f"✅ SLA: Data Freshness - {sla_freshness} ({round(hours_old, 1)} hours old)")

# SLA 2: Data Completeness (should have all repos)
expected_repos = 10
actual_repos = gold_repos.count()

sla_completeness = "PASS" if actual_repos >= expected_repos else "FAIL"
print(f"✅ SLA: Data Completeness - {sla_completeness} ({actual_repos}/{expected_repos} repos)")

# SLA 3: Quality Metrics (95% pass rate required)
quality_pass_rate = 95.0  # From previous quality checks

sla_quality = "PASS" if quality_pass_rate >= 95.0 else "FAIL"
print(f"✅ SLA: Quality Metrics - {sla_quality} ({round(quality_pass_rate, 2)}% pass rate)")

# SLA 4: Performance (should complete in < 5 minutes)
# Note: In real scenario, track actual execution time
estimated_runtime = 2.5  # minutes

sla_performance = "PASS" if estimated_runtime < 5.0 else "FAIL"
print(f"✅ SLA: Performance - {sla_performance} (≈{estimated_runtime} minutes)")


In [0]:

# Cell 9: Store Monitoring Metrics
metrics_data = [
    ("M001", "total_stars", total_stars, "count", "business", datetime.now(), "pipeline_001", "OK"),
    ("M002", "avg_health_score", avg_health, "score", "quality", datetime.now(), "pipeline_001", "OK"),
    ("M003", "total_contributors", float(total_contributors), "count", "business", datetime.now(), "pipeline_001", "OK"),
    ("M004", "expert_contributors", float(expert_contribs), "count", "business", datetime.now(), "pipeline_001", "OK"),
    ("M005", "avg_stars_per_repo", avg_stars, "count", "performance", datetime.now(), "pipeline_001", "OK"),
    ("M006", "data_freshness_hours", hours_old, "hours", "sla", datetime.now(), "pipeline_001", sla_freshness),
    ("M007", "data_completeness_percent", float(actual_repos)/expected_repos*100, "percent", "sla", datetime.now(), "pipeline_001", sla_completeness),
    ("M008", "quality_pass_rate", quality_pass_rate, "percent", "sla", datetime.now(), "pipeline_001", sla_quality),
]

metrics_df = spark.createDataFrame(
    metrics_data,
    ["metric_id", "metric_name", "metric_value", "metric_unit", "metric_category", 
     "metric_timestamp", "pipeline_run_id", "status"]
)

metrics_df.write.format("delta").mode("append").option("mergeSchema", "true").insertInto(
    f"{catalog}.{schema}.pipeline_metrics"
)

print("\n✅ Monitoring metrics saved")
